# M8.2 · NeMo Evaluator — the art of the possible (~30 min)

M8.1 set up the question every model build must answer: **"how do you know your
model actually got better?"** This hands-on shows NeMo Evaluator answering it,
reusing the installed **`nemo-evaluator`** package (the `nel` CLI) — no wheel
reinvented.

We walk the **three evaluation modes** from the deck against an OpenAI-compatible
endpoint (here, NVIDIA-hosted models on `build.nvidia.com`):

| Mode | What it answers | How |
|------|-----------------|-----|
| **1. Academic benchmarks** | general capability (reasoning, knowledge) | `nel eval run --bench gsm8k` |
| **2. The bake-off** | *did my model get better?* | run the same benchmark across **two models** and compare with `nel eval report` |
| **3. LLM-as-judge** | task-specific quality on **your** rubric | grade AML SAR narratives with a judge model (ties to M8.1 slide 6) |

**Engine:** the installed `nemo-evaluator` (v0.3.x) exposes the **`nel`** CLI. It
pulls each benchmark's dataset, runs it against any OpenAI-compatible endpoint via
the seed → solve → verify loop, and writes per-sample results + a score bundle.
Reference: `workshop/reference/repos/Evaluator`.

## 1. Prerequisites (inline)

`nemo-evaluator` is already in this environment. We only ensure `openai` (for the
Mode-3 judge) and collect your **NVIDIA API key** (from
[build.nvidia.com](https://build.nvidia.com), starts `nvapi-`). No GPU needed —
evaluation calls a hosted endpoint over HTTP.

In [1]:
import os, getpass, subprocess, sys
from pathlib import Path

# Per-notebook uv venv (.venv/ in this folder).
sys.path.insert(0, str(Path.cwd().parent))
from notebook_env import bootstrap_notebook_env, ensure

bootstrap_notebook_env()

ensure("nemo_evaluator", ["nemo-evaluator"], quiet=True)
ensure("openai", ["openai>=1.40.0"], quiet=True)

import nemo_evaluator
print("nemo_evaluator version:", nemo_evaluator.__version__)

# NVIDIA API key (runtime prompt; not stored in the notebook). The nel CLI reads
# NVIDIA_API_KEY / NEMO_API_KEY from the environment.
if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA API key (nvapi-...): ").strip()
os.environ.setdefault("NEMO_API_KEY", os.environ["NVIDIA_API_KEY"])
assert os.environ.get("NVIDIA_API_KEY"), "NVIDIA_API_KEY required to call the hosted endpoint."

MODEL_URL = "https://integrate.api.nvidia.com/v1"   # /chat/completions is auto-appended
API_KEY = os.environ["NVIDIA_API_KEY"]
print("\nEndpoint:", MODEL_URL)

# Helper: invoke the `nel` CLI in-process via python -m (robust to PATH).
def nel(*args, timeout=1800):
    cmd = [sys.executable, "-m", "nemo_evaluator.cli.main", *map(str, args)]
    print("$", "nel", *args)
    p = subprocess.run(cmd, text=True, capture_output=True, timeout=timeout)
    if p.stdout: print(p.stdout)
    if p.returncode != 0:
        print("[stderr]\n", p.stderr[-2000:])
    return p


creating uv venv: /home/ubuntu/repo-content_copy/workshop-Materials/M8-nemo_evaluator/.venv
notebook env: /home/ubuntu/repo-content_copy/workshop-Materials/M8-nemo_evaluator/.venv (python /home/ubuntu/repo-content_copy/workshop-Materials/M8-nemo_evaluator/.venv/bin/python)
installing: nemo_evaluator ...


Using CPython 3.12.13
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


installing: openai ...
nemo_evaluator version: 0.3.0


NVIDIA API key (nvapi-...):  ········



Endpoint: https://integrate.api.nvidia.com/v1


## 2. What can it evaluate? — list built-in benchmarks

NeMo Evaluator ships many academic benchmarks (GSM8K, MMLU / MMLU-Pro, HumanEval,
GPQA, MATH-500, SimpleQA, …). Each is a registered *environment* you can run
against your endpoint with `nel eval run --bench <name>`.

In [2]:
from nemo_evaluator.environments.registry import list_environments
names = list_environments()
print(f"{len(names)} built-in benchmarks:\n")
print(", ".join(names))


17 built-in benchmarks:

drop, gpqa, gsm8k, healthbench, humaneval, math500, mgsm, mmlu, mmlu_pro, nmp_harbor, pinchbench, simpleqa, terminal-bench-hard, terminal-bench-hard-aa-split, terminal-bench-v1, triviaqa, xstest


## 3. Mode 1 — Academic benchmark (GSM8K)

Run GSM8K (grade-school math word problems) against one hosted model. The `nel`
quick mode needs **what** to run (`--bench`) and **where** (`--model-url` +
`--model-id` + `--api-key`).

**Flag significance:**
- `--bench gsm8k` — the benchmark id (from the list above).
- `--model-url` — OpenAI-compatible base; `/chat/completions` is auto-appended.
- `--model-id` — which hosted model to evaluate.
- `--api-key` — bearer token for the endpoint (your `nvapi-` key).
- `--max-problems 5` — **the knob that makes this fit in 30 min.** Caps the number
  of questions; drop it for the full benchmark (the real, citable number).
- `--temperature 0.0` — deterministic decoding; benchmarks want the single best answer.
- `-o` — output directory (per-sample `results.jsonl` + an `eval-*.json` score bundle).

In [3]:
MODEL_A = "meta/llama-3.2-3b-instruct"

nel("eval", "run", "--bench", "gsm8k",
    "--model-url", MODEL_URL, "--model-id", MODEL_A, "--api-key", API_KEY,
    "--max-problems", 5, "--temperature", 0.0,
    "-o", "./eval_results/gsm8k_3b")

# Render the score bundle as a table.
nel("eval", "report", "./eval_results/gsm8k_3b", "-f", "markdown")


$ nel eval run --bench gsm8k --model-url https://integrate.api.nvidia.com/v1 --model-id meta/llama-3.2-3b-instruct --api-key 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 --max-problems 5 --temperature 0.0 -o ./eval_results/gsm8k_3b

  Benchmark 1/1: gsm8k

  gsm8k: pass@1=0.8000 
Report: eval_results/gsm8k_3b/report.md

Completed 1 benchmark(s). Results: ./eval_results/gsm8k_3b

$ nel eval report ./eval_results/gsm8k_3b -f markdown
# Evaluation Report: meta/llama-3.2-3b-instruct

| Benchmark | Scorer | Samples | Score | CI (95%) |
|-----------|--------|---------|-------|----------|
| gsm8k | pass@1 | 5 | 0.8000 |  |




CompletedProcess(args=['/home/ubuntu/repo-content_copy/workshop-Materials/M8-nemo_evaluator/.venv/bin/python', '-m', 'nemo_evaluator.cli.main', 'eval', 'report', './eval_results/gsm8k_3b', '-f', 'markdown'], returncode=0, stdout='# Evaluation Report: meta/llama-3.2-3b-instruct\n\n| Benchmark | Scorer | Samples | Score | CI (95%) |\n|-----------|--------|---------|-------|----------|\n| gsm8k | pass@1 | 5 | 0.8000 |  |\n\n', stderr='')

## 4. Mode 2 — The bake-off ("did my model get better?")

The core eval value-prop: run the **same** benchmark across **two models** and
compare. In production you'd compare *base vs SFT vs DPO* (your M7 checkpoints);
here we compare two hosted models (3B vs 8B) so it runs anywhere. Write both runs
under one parent directory, then a single `nel eval report` builds the comparison
table across every score bundle it finds.

In [4]:
BAKEOFF = "./eval_results/bakeoff"
MODELS = ["meta/llama-3.2-3b-instruct", "meta/llama-3.1-8b-instruct"]

for m in MODELS:
    nel("eval", "run", "--bench", "gsm8k",
        "--model-url", MODEL_URL, "--model-id", m, "--api-key", API_KEY,
        "--max-problems", 5, "--temperature", 0.0,
        "-o", f"{BAKEOFF}/{m.replace('/', '_')}")

# One report over the parent dir = side-by-side comparison of both models.
nel("eval", "report", BAKEOFF, "-f", "markdown")
print("With the FULL dataset this table is exactly how you decide whether a new")
print("checkpoint beat the previous one.")


$ nel eval run --bench gsm8k --model-url https://integrate.api.nvidia.com/v1 --model-id meta/llama-3.2-3b-instruct --api-key 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 --max-problems 5 --temperature 0.0 -o ./eval_results/bakeoff/meta_llama-3.2-3b-instruct

  Benchmark 1/1: gsm8k

  gsm8k: pass@1=0.8000 
Report: eval_results/bakeoff/meta_llama-3.2-3b-instruct/report.md

Completed 1 benchmark(s). Results: ./eval_results/bakeoff/meta_llama-3.2-3b-instruct

$ nel eval run --bench gsm8k --model-url https://integrate.api.nvidia.com/v1 --model-id meta/llama-3.1-8b-instruct --api-key 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 --max-problems 5 --temperature 0.0 -o ./eval_results/bakeoff/meta_llama-3.1-8b-instruct

  Benchmark 1/1: gsm8k

  gsm8k: pass@1=1.0000 
Report: eval_results/bakeoff/meta_llama-3.1-8b-instruct/report.md

Completed 1 benchmark(s). Results: ./eval_results/bakeoff/meta_llama-3.1-8b-instruct

$ nel eval report ./eval_res

## 5. Mode 3 — LLM-as-judge on your own rubric (AML SAR)

Academic benchmarks don't measure *your* task. For that you use an **LLM-as-judge**:
a strong model grades outputs against a rubric. This mirrors M8.1 slide 6 — the
AML SAR rubric (groundedness, statute correctness, objective tone).

NeMo Evaluator productionizes this via judge-based scorers (`needs_judge`) and
**BYOB** custom benchmarks (`@benchmark` + `@scorer`, run with
`nel eval run --bench ./my_bench.py`). Below is a compact, framework-agnostic
illustration of the same pattern so the score gap is visible in seconds.

> **Calibration caveat (M8.1 slide 7):** the judge is itself a model — validate
> its scores against human labels before trusting them, and use temperature 0 so
> grades are reproducible.

In [5]:
import json, re
from openai import OpenAI

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=API_KEY)
JUDGE_MODEL = "meta/llama-3.1-8b-instruct"   # the grader (use a strong model)

# Two candidate SAR narratives for the SAME case — one regulator-grade, one not.
CASE = "Entity SYN_X made three cash deposits of $9,793, $9,807, $9,791 over two days to AC501879."
CANDIDATES = {
    "good": "The pattern of three cash deposits just below the $10,000 CTR threshold is consistent with structuring under 31 U.S.C. 5324; the activity warrants review and is deferred to the human investigator.",
    "bad": "This is definitely money laundering. The customer is obviously a criminal and should be reported immediately.",
}

RUBRIC = (
    "You are an AML compliance reviewer. Score the SAR narrative from 0-1 on each of: "
    "groundedness (claims tied to the case facts), statute_correct (cites the right law, e.g. 31 U.S.C. 5324), "
    "tone (objective, qualified language; penalise 'definitely'/'obviously'/'criminal'). "
    "Return ONLY JSON: {\"groundedness\":x, \"statute_correct\":x, \"tone\":x}."
)

def judge(narrative):
    r = client.chat.completions.create(
        model=JUDGE_MODEL, temperature=0.0, max_tokens=120,
        messages=[{"role": "system", "content": RUBRIC},
                  {"role": "user", "content": f"Case: {CASE}\n\nSAR narrative: {narrative}"}])
    txt = r.choices[0].message.content
    m = re.search(r"\{[^}]+\}", txt)
    return json.loads(m.group()) if m else {"raw": txt}

for label, narrative in CANDIDATES.items():
    print(f"{label:5s}: {judge(narrative)}")
print("\nThe judge should score the regulator-grade narrative higher on tone +")
print("statute_correct — the same signal that drove the M7.4 DPO preference data.")


good : {'groundedness': 0.9, 'statute_correct': 1, 'tone': 0.9}
bad  : {'groundedness': 0, 'statute_correct': 0, 'tone': 0}

The judge should score the regulator-grade narrative higher on tone +
statute_correct — the same signal that drove the M7.4 DPO preference data.


## 6. Where to go next

- **More benchmarks:** `list_environments()` above; add `lm-eval://<task>` or
  `skills://<task>` sources (`nel list --source all`).
- **Custom benchmarks (BYOB):** define `@benchmark` + `@scorer` in a `.py` file and
  run `nel eval run --bench ./my_bench.py` — including judge-based scorers
  (`needs_judge`) for rubric grading like Mode 3.
- **Orchestration & scale:** `nel eval run` also supports `--background`, `--submit`
  (Slurm), and report export (`nel eval report -f csv/html/json`).
- **The bake-off in practice:** point Mode 2 at your **M7** base/SFT/DPO models
  (deploy each behind a NIM — see M9) to measure real training progress.